In [ ]:
! pip install -q transformers datasets evaluate accelerate huggingface_hub

There are two common types of question answering tasks:

- Extractive: extract the answer from the given context.
- Abstractive: generate an answer from the context that correctly answers the question.

This guide will show you how to:

1. Finetune [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) on the [SQuAD](https://huggingface.co/datasets/squad) dataset for extractive question answering.
2. Use your finetuned model for inference.

## Load dependencies

In [ ]:
import pprint
import torch
import numpy as np
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import (
AutoTokenizer, AutoModelForQuestionAnswering,
pipeline, AutoModelForQuestionAnswering,
TrainingArguments, Trainer, DefaultDataCollator
)

## Config

In [ ]:
user_name = "amin-oj"
dataset_name = "squad"
model_name = "distilbert/distilbert-base-uncased"
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 3
ckpt_name = f"{model_name.split("/")[-1]}-finetuned-qa"

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
squad = load_dataset(dataset_name, split="train[:5000]")
squad = squad.train_test_split(test_size=0.2)
squad["train"][0]

There are several important fields here:

- `answers`: the starting location of the answer token and the answer text.
- `context`: background information from which the model needs to extract the answer.
- `question`: the question a model should answer.

## Preprocess

Preprocessing steps particular to question answering tasks:

1. Some examples in a dataset may have a very long `context` that exceeds the maximum input length of the model. To deal with longer sequences, truncate only the `context` by setting `truncation="only_second"`.
2. Next, map the start and end positions of the answer to the original `context` by setting
   `return_offset_mapping=True`.
3. With the mapping in hand, now you can find the start and end tokens of the answer. Use the [sequence_ids](https://huggingface.co/docs/tokenizers/main/en/api/encoding#tokenizers.Encoding.sequence_ids) method to
   find which part of the offset corresponds to the `question` and which corresponds to the `context`.

In [ ]:
def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,# return `(char_start, char_end)` for each token?
        padding="max_length",
        return_tensors = "np"
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
        # a list mapping the tokens to the id of their original sentences:
            # - `None` for special tokens added around or between sequences
            # - `0` for tokens corresponding to words in the first sequence (question here)
            # - `1` for tokens corresponding to words in the second sequence (context here)

        # Find the start and end of the context
        if 1 not in sequence_ids:
            start_positions.append(0)
            end_positions.append(0)
            continue
        context_start = sequence_ids.index(1)
        context_end = sequence_ids.index(None, context_start) -1

        # If the answer is not fully inside the context, label it (0, 0)
        if (offset[context_start][0] > end_char) or (offset[context_end][1] < start_char):
            # The first condition never happens. Do you know why?
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_position = context_start + np.argmax(offset[context_start:context_end+1, 0] == start_char)
            start_positions.append(start_position)
            end_position = context_start + np.argmax(offset[context_start:context_end+1, 1] == end_char)
            end_positions.append(end_position)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_squad = squad.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    remove_columns=squad["train"].column_names,
    fn_kwargs = {"tokenizer": tokenizer}
)

## Train

In [ ]:
data_collator = DefaultDataCollator()
# Unlike other data collators in HF Transformers, the DefaultDataCollator
# does not apply any additional preprocessing such as padding.

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir=ckpt_name,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    push_to_hub=push_to_hub,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [ ]:
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Evaluate

In [ ]:
# https://huggingface.co/learn/llm-course/chapter7/7?fw=pt#post-processing

## Inference

### Simplest way

In [ ]:
question_answerer = pipeline("question-answering", model=f"{user_name}/{ckpt_name}")
question = "How many programming languages does BLOOM support?"
context = "BLOOM has 176 billion parameters and can generate text in 46 languages natural languages and 13 programming languages."
question_answerer(question=question, context=context)

### More complex | More flexible

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(question, context, return_tensors="pt")
model = AutoModelForQuestionAnswering.from_pretrained(f"{user_name}/{ckpt_name}")
with torch.no_grad():
    outputs = model(**inputs)
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)